# Lesson 04 - Wzorzec projektowy Użycia Narzędzi

W tej lekcji poznasz wzorzec projektowy **Użycia Narzędzi** dla agentów AI z wykorzystaniem Microsoft Agent Framework (Python). Omówimy:

- Definiowanie narzędzi jako funkcji za pomocą dekoratora `@tool` oraz typowanych parametrów
- Dostarczanie schematów narzędzi, aby model wiedział, do czego każde narzędzie służy
- Kontrolowanie wykonania narzędzi przez `approval_mode`
- Zwracanie **ustrukturyzowanych wyników** za pomocą modeli Pydantic oraz `response_format`

Scenariusz to **agent do rezerwacji podróży**, który może wyszukiwać destynacje, sprawdzać dostępność oraz pobierać informacje o lotach.


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definiowanie narzędzi za pomocą dekoratora @tool

Dekorator `@tool` zamienia zwykłą funkcję Pythona w narzędzie, które może wywołać agent.
Kluczowe punkty:

- **Dokumentacja** staje się opisem narzędzia, który widzi model.
- **Adnotacje typów** (w tym `Annotated` z opisami) definiują schemat narzędzia.
- `approval_mode` kontroluje, czy użytkownik musi zatwierdzić każde wywołanie przed jego wykonaniem.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Tworzenie agenta z wieloma narzędziami

Przekaż wszystkie trzy narzędzia do klienta, aby model mógł wywołać te, których potrzebuje do odpowiedzi na pytanie użytkownika.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturalny wynik z narzędziami

Ustawiając `response_format` na model Pydantic, agent jest zmuszony do zwrócenia dobrze typowanego obiektu JSON zamiast swobodnego tekstu. Jest to przydatne, gdy dalszy kod musi programowo przetwarzać wynik.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Wzorce zatwierdzania narzędzi

Parametr `approval_mode` w `@tool` kontroluje, czy wywołania narzędzi wymagają zatwierdzenia przez człowieka przed wykonaniem:

| Tryb | Zachowanie |
|---|---|
| `"never_require"` | Narzędzie działa automatycznie — nie jest potrzebne potwierdzenie użytkownika. |
| `"always_require"` | Każde wywołanie musi zostać zatwierdzone przez użytkownika przed wykonaniem. |

Używaj `"always_require"` dla narzędzi, które mają skutki uboczne (np. rezerwacja lotu, obciążenie karty kredytowej), aby człowiek pozostawał w pętli.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Podsumowanie

W tej lekcji nauczyłeś się, jak:

1. **Definiować narzędzia** za pomocą dekoratora `@tool` z typowanymi parametrami i docstringami, które służą jako schemat narzędzia.
2. **Kompilować wiele narzędzi**, aby agent mógł wywoływać je w kolejności, odpowiadając na złożone zapytania.
3. **Zwracać ustrukturyzowane wyniki** przez przekazanie modelu Pydantic jako `response_format`.
4. **Kontrolować zatwierdzanie narzędzi** za pomocą `approval_mode`, aby utrzymać człowieka w pętli dla wrażliwych operacji.

Te wzorce stanowią podstawę do budowania niezawodnych, gotowych do produkcji agentów, którzy mogą bezpiecznie współdziałać z systemami zewnętrznymi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
